Config

In [ ]:
setup = """
set -e

sudo apt update
sudo apt install -y build-essential git wget ca-certificates

git clone https://github.com/Decrescent398/arc-agi-3-solution.git /repo
cd /repo/ARC3-Inference
chmod +x scripts/*.sh

uv python install 3.12.12
uv venv --python 3.12.12 .venv
uv lock --python 3.12.12
make install

cd /tmp
wget https://developer.download.nvidia.com/compute/cuda/12.8.1/local_installers/cuda_12.8.1_570.124.06_linux.run
sudo sh cuda_12.8.1_570.124.06_linux.run --silent --toolkit --toolkitpath=/opt/cuda-12.8 --override
test -x /opt/cuda-12.8/bin/nvcc
/opt/cuda-12.8/bin/nvcc --version

uv pip install --python /repo/ARC3-Inference/.venv/bin/python nvidia-cuda-runtime-cu12==12.8.* nvidia-cuda-nvcc-cu12==12.8.* nvidia-cuda-nvrtc-cu12==12.8.* nvidia-cuda-cupti-cu12==12.8.* nvidia-cuda-cccl-cu12

export CUDA_HOME=/opt/cuda-12.8
export CUDA_PATH=/opt/cuda-12.8
export VIRTUAL_ENV=/repo/ARC3-Inference/.venv

export PATH="/opt/cuda-12.8/bin:$VIRTUAL_ENV/bin:$PATH"
export LD_LIBRARY_PATH=/opt/cuda-12.8/lib64
export LIBRARY_PATH=/opt/cuda-12.8/lib64

CCCL="$VIRTUAL_ENV/lib/python3.12/site-packages/nvidia/cuda_cccl/include"

test -d "$CCCL"
mkdir -p /tmp/arc3-cuda-includes
ln -sfn "$CCCL" /tmp/arc3-cuda-includes/cccl

export CPATH="/opt/cuda-12.8/include:/tmp/arc3-cuda-includes:$CCCL"
"""

Setup

In [ ]:
import os
import subprocess
from pathlib import Path

In [ ]:
subprocess.run(["bash", "-c", setup], check=True)

venv = Path("/repo/ARC3-Inference/.venv")
cuda = Path("/opt/cuda-12.8")
cccl = venv / "lib/python3.12/site-packages/nvidia/cuda_cccl/include"

compat = Path("/tmp/arc3-cuda-includes")
compat.mkdir(parents=True, exist_ok=True)
(compat / "cccl").symlink_to(cccl, target_is_directory=True) if not (compat / "cccl").exists() else None

env = os.environ.copy()
env.update({
    "VIRTUAL_ENV": str(venv),
    "CUDA_HOME": str(cuda),
    "CUDA_PATH": str(cuda),
    "PATH": f"{venv}/bin:{cuda}/bin:{env['PATH']}",
    "CPATH": f"{cuda}/include:{compat}:{cccl}",
    "LIBRARY_PATH": f"{cuda}/lib64",
    "LD_LIBRARY_PATH": f"{cuda}/lib64",
})

In [ ]:
try:
    os.mkdir("results")
except FileExistsError:
    pass

In [ ]:
d = "/repo/ARC3-Inference"
game = "tn36-ef4dde99"
n_passes = 1
max_runtime_mins = 10 

Execute

```console
tail -f /repo/ARC3-Inference/.cache/arc3_runtime/arc3-inference-server.log
```

In [ ]:
def run():
    try:
        subprocess.run(
            f"cd {d} && uv run --no-sync --python 3.12.12 python -c \"import arc_agi; arc_agi.Arcade(environments_dir='/results/envs').make('{game}')\"",
            check=True, shell=True, env=env)

        subprocess.run(
            f"cd {d} && make server && make interactive GAME={game} N_PASSES={n_passes} "
            f"MAX_RUNTIME_MINUTES={max_runtime_mins} ENVIRONMENTS_DIR=/results/envs EXPERIMENTS_DIR=/results",
            check=True, shell=True, env=env)
    finally:
        pass

In [ ]:
def list_games():
    subprocess.run(f"cd {d} && uv run --no-sync inference-taaf-run --include-tags official --list-games", shell=True, check=True)